# 03 - Descriptive tables

Descriptive tables, written to `results/`:

1. **Table 4** - societal actor composition (four function categories), per country x issue.
2. **Table 5** - political actor composition (government vs opposition), per country x issue,
   the companion to Table 4.
3. **Political balance** - government/opposition share per country with the reflective benchmark.
4. **Diversity by period** - societal and political Dz in campaign vs routine weeks (supplementary
   to the campaign ANOVAs in Tables 6-7).

Composition uses the within-article percentage averaged across articles (as in the manuscript's
Table 4); the diversity scores come from `02`.

In [ ]:
import numpy as np
import pandas as pd

import helpers as h

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

soc = pd.read_csv(h.SOCIAL_CLEAN, sep=h.CSV_SEP)
pol = pd.read_csv(h.POLITICAL_CLEAN, sep=h.CSV_SEP)
soc["topic_label"] = soc["topic"].map(h.TOPIC_LABEL).fillna(soc["topic"])
pol["topic_label"] = pol["topic"].map(h.TOPIC_LABEL).fillna(pol["topic"])
refl = h.reflective_gov_share(pol)   # Series indexed by country
print(f"societal articles: {len(soc):,} | political actor-rows: {len(pol):,}")

## 1. Table 3 - societal actor composition (country x issue)

For each article with at least one actor, the within-article percentage of each category, then
averaged across articles. Rows sum to ~100%. `M_actors` is the mean actors per article.

In [ ]:
CAT_PCT = {"nr_politicians": "Politicians", "nr_experts_professionals": "Experts/professionals",
           "nr_civil_society": "Civil society", "nr_general_public": "General public"}

d = soc[soc["nr_actors_total"] >= 1].copy()
for c in h.CAT_COLS:
    d[c + "_p"] = 100 * d[c] / d["nr_actors_total"]

def table3(group_keys):
    agg = {"N": ("article_id", "size"), "M_actors": ("nr_actors_total", "mean")}
    agg.update({CAT_PCT[c]: (c + "_p", "mean") for c in h.CAT_COLS})
    t = d.groupby(group_keys).agg(**agg)
    cols = ["N"] + list(CAT_PCT.values()) + ["M_actors"]
    return t[cols].round(1)

table3_ct = table3(["country", "topic_label"])
table3_c  = table3(["country"])
table3_ct.to_csv(h.TABLES_DIR / "Table3_societal_composition.csv")
print(f"N articles with >=1 actor: {len(d):,}")
table3_ct

## 2. Table 3 - political actor composition (country x issue)

Companion to Table 4 for political actors: the mean within-article percentage of governing- and
opposition-party actors, per country and issue, with the mean number of party-linked political
actors per article. Built the same way as Table 4 (per-article percentages averaged).

In [ ]:
pol["govw"] = np.where(pol["government"] == 1, pol["nr_actors"].fillna(1), 0)
pol["oppw"] = np.where(pol["government"] == 0, pol["nr_actors"].fillna(1), 0)
art = (pol.groupby(["article_id", "country", "topic"])
          .agg(gov=("govw", "sum"), opp=("oppw", "sum")).reset_index())
art["total"] = art["gov"] + art["opp"]
art = art[art["total"] > 0]
art["pgov"] = 100 * art["gov"] / art["total"]
art["popp"] = 100 * art["opp"] / art["total"]
art["topic_label"] = art["topic"].map(h.TOPIC_LABEL).fillna(art["topic"])
comp = art.groupby(["country", "topic_label"]).agg(
    N=("article_id", "size"), Government=("pgov", "mean"),
    Opposition=("popp", "mean"), M=("total", "mean"))

rows = []
for cc in h.COUNTRY_ORDER:
    for k, iss in enumerate(["inflation", "immigration"]):
        s = comp.loc[(cc, iss)]
        rows.append({"Country": h.COUNTRY_NAME[cc] if k == 0 else "", "Issue": iss.capitalize(),
                     "N": f"{int(s['N']):,}", "Government": f"{s['Government']:.1f}",
                     "Opposition": f"{s['Opposition']:.1f}", "M actors/article": f"{s['M']:.1f}"})
pol_comp = pd.DataFrame(rows)
print(pol_comp.to_string(index=False))

pol_comp.to_csv(h.TABLES_DIR / "Table3_political_composition.csv", index=False)

## 2b. Combined composition table (societal + political in one)

Tables 4 and 5 merged into a single wide table with column-group headers. Use this in place of the
two separate tables if you prefer one big table. Societal and political columns keep separate N and
M because they have different denominators (any societal actor vs any party-linked political actor).

In [ ]:
soc_t = d.groupby(["country", "topic_label"]).agg(
    Nsoc=("article_id", "size"), Politicians=("nr_politicians_p", "mean"),
    Experts=("nr_experts_professionals_p", "mean"), Civil=("nr_civil_society_p", "mean"),
    General=("nr_general_public_p", "mean"), Msoc=("nr_actors_total", "mean"))

rows = []
for cc in h.COUNTRY_ORDER:
    for k, iss in enumerate(["inflation", "immigration"]):
        s = soc_t.loc[(cc, iss)]; p = comp.loc[(cc, iss)]
        rows.append([h.COUNTRY_NAME[cc] if k == 0 else "", iss.capitalize(),
                     f"{int(s.Nsoc):,}", f"{s.Politicians:.1f}", f"{s.Experts:.1f}",
                     f"{s.Civil:.1f}", f"{s.General:.1f}", f"{s.Msoc:.1f}",
                     f"{int(p.N):,}", f"{p.Government:.1f}", f"{p.Opposition:.1f}", f"{p.M:.1f}"])
combined = pd.DataFrame(rows, columns=["Country", "Issue", "N", "Politicians", "Experts/prof.",
                                       "Civil society", "General public", "M",
                                       "N", "Government", "Opposition", "M"])
print(combined.to_string(index=False))

h.apa_table(combined, number=3,
            title="Composition of Societal and Political Actors per Country and Issue",
            path=h.TABLES_DIR / "Table3_combined_composition.docx",
            spanners=[("", 2), ("Societal actors", 6), ("Political actors", 4)], landscape=True,
            note=("Societal N = articles with at least one societal actor; political N = articles "
                  "with at least one party-linked political actor. Entries are the mean across "
                  "articles of the within-article percentage of actors in each group. M = mean "
                  "number of actors per article (societal and political, respectively)."))
combined.to_csv(h.TABLES_DIR / "Table3_combined_composition.csv", index=False)

## 3. Political balance - government vs opposition

Share of political actors (weighted by actor count) that are governing- vs opposition-party,
per country and by topic. `dev_reflective` = governing share minus government seat share
(> 0 = governing parties over-represented relative to their seats).

In [ ]:
pol["w"] = pol["nr_actors"].fillna(1)

def balance(group_keys):
    g = (pol.groupby(group_keys + ["government"])["w"].sum()
            .unstack("government").rename(columns={1: "gov", 0: "opp"}).fillna(0))
    g["n_actors"] = (g["gov"] + g["opp"]).astype(int)
    g["pct_gov"] = (100 * g["gov"] / g["n_actors"]).round(1)
    g["pct_opp"] = (100 * g["opp"] / g["n_actors"]).round(1)
    cc = g.index.get_level_values("country")
    g["seat_share_gov"] = (100 * refl.reindex(cc).values).round(1)
    g["dev_reflective"] = (g["pct_gov"] - g["seat_share_gov"]).round(1)
    return g[["n_actors", "pct_gov", "pct_opp", "seat_share_gov", "dev_reflective"]]

balance_c  = balance(["country"]).reindex(h.COUNTRY_ORDER)
balance_ct = balance(["country", "topic_label"])
balance_c.to_csv(h.TABLES_DIR / "Table_political_balance.csv")
balance_ct.to_csv(h.TABLES_DIR / "Table_political_balance_topic.csv")
print("Per country:")
print(balance_c.to_string())
print("\nPer country x topic:")
balance_ct

## 4. Diversity by period (campaign vs routine, supplementary to Tables 6-7)

Mean weekly Simpson's Dz (outlet-averaged, from `02`) in campaign vs non-campaign weeks, per
country, for societal and political diversity. Supports H3a (societal declines in campaign) and
H3b (political rises, except Turkey).

In [ ]:
soc_cw = pd.read_csv(h.RESULTS_DIR / "soc_div_country_week.csv")
pol_cw = pd.read_csv(h.RESULTS_DIR / "pol_div_country_week.csv")
PERIOD = {0: "non_campaign", 1: "campaign"}

def by_period(div, label):
    g = div.groupby(["country", "campaign_period"])["simpson_Dz"].mean().unstack("campaign_period")
    g.columns = [f"{label}_{PERIOD[c]}" for c in g.columns]
    g[f"{label}_overall"] = div.groupby("country")["simpson_Dz"].mean()
    g[f"{label}_delta"] = g[f"{label}_campaign"] - g[f"{label}_non_campaign"]
    return g.round(3)

table5 = pd.concat([by_period(soc_cw, "societal"), by_period(pol_cw, "political")], axis=1)
table5 = table5.reindex(h.COUNTRY_ORDER)
table5.to_csv(h.TABLES_DIR / "diversity_by_period_summary.csv")
print("Dz by period (delta = campaign - non-campaign):")
table5[["societal_non_campaign", "societal_campaign", "societal_delta",
        "political_non_campaign", "political_campaign", "political_delta"]]